# LeetCode 2889 — Reshape Data: Pivot bằng Python (pandas)

Link đề: https://leetcode.com/problems/reshape-data-pivot/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def pivotTable(weather: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `weather` ở dạng **"dài"** (mỗi dòng là một cặp `city`, `month` với một `temperature`); **xoay (pivot)** thành bảng **"rộng"**: mỗi **hàng là một tháng**, mỗi **cột là một thành phố**, ô là nhiệt độ.
2. Phân biệt **dữ liệu dạng dài (long)** và **dạng rộng (wide)**, và biết `pivot` là phép biến đổi dài -> rộng.
3. Đi từ cách **vòng lặp xây từ điển lồng nhau** → **`pivot(index, columns, values)`** → **`pivot_table(..., aggfunc=...)`** → **`set_index(...).unstack()`** → **`groupby(...).sum().unstack()`**.
4. Hiểu **index và tên cột** của kết quả pivot (`month` thành index, `city` thành tên các cột), thứ tự **sắp xếp theo bảng chữ cái**, và khi nào cần `reset_index()`.
5. Tránh các lỗi kinh điển: `pivot` với dữ liệu **trùng cặp** (báo lỗi), truyền tham số theo vị trí, nhầm `index` với `columns`, quên rằng tháng bị xếp theo chữ cái chứ không theo thời gian.

## 2. Đề bài & ví dụ

Có một bảng `weather` với các cột `city`, `month`, `temperature`. Mỗi cặp (`city`, `month`) xuất hiện **tối đa một lần**.

Hãy viết hàm **xoay bảng** sao cho **mỗi hàng ứng với một `month`**, **mỗi cột ứng với một `city`**, và giá trị trong ô là `temperature`.

**Ví dụ**

- Input: DataFrame `weather`

| city | month | temperature |
|---|---|---|
| Jacksonville | January | 13 |
| Jacksonville | February | 23 |
| Jacksonville | March | 38 |
| Jacksonville | April | 5 |
| Jacksonville | May | 34 |
| ElPaso | January | 20 |
| ElPaso | February | 6 |
| ElPaso | March | 26 |
| ElPaso | April | 2 |
| ElPaso | May | 43 |

- Output:

| month | ElPaso | Jacksonville |
|---|---|---|
| April | 2 | 5 |
| February | 6 | 23 |
| January | 20 | 13 |
| March | 26 | 38 |
| May | 43 | 34 |

- Giải thích: 10 dòng dài thành bảng `5 x 2`. Tháng và thành phố được xếp theo **thứ tự chữ cái** (April, February, January, March, May; ElPaso, Jacksonville), không theo thứ tự thời gian.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc. Việc `month` nằm ở **index** hay ở **một cột thường** (sau `reset_index()`) của kết quả cũng nên đối chiếu với đề gốc; notebook này dùng `pivot` chuẩn, `month` là index.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

weather = pd.DataFrame({
    "city": ["Jacksonville"] * 5 + ["ElPaso"] * 5,
    "month": ["January", "February", "March", "April", "May"] * 2,
    "temperature": [13, 23, 38, 5, 34, 20, 6, 26, 2, 43],
})
print(weather)
print()
print(weather.dtypes)
print("pandas", pd.__version__)

           city     month  temperature
0  Jacksonville   January           13
1  Jacksonville  February           23
2  Jacksonville     March           38
3  Jacksonville     April            5
4  Jacksonville       May           34
5        ElPaso   January           20
6        ElPaso  February            6
7        ElPaso     March           26
8        ElPaso     April            2
9        ElPaso       May           43

city             str
month            str
temperature    int64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Dữ liệu dạng dài (long) và dạng rộng (wide)

- **Dạng dài:** mỗi dòng là **một quan sát** (ở đây: nhiệt độ của một thành phố trong một tháng). Có ít cột, nhiều dòng. Bảng `weather` là dạng dài.
- **Dạng rộng:** mỗi **giá trị của một biến** (ở đây: từng thành phố) trở thành **một cột**. Có nhiều cột, ít dòng. Dễ đọc cho người, dễ so sánh các thành phố cạnh nhau.

`pivot` biến **dài -> rộng**. Phép ngược lại là `melt` (bài 2890).

In [2]:
print("dạng dài :", weather.shape, "(10 dòng, 3 cột)")
print("số thành phố:", weather["city"].nunique(), "| số tháng:", weather["month"].nunique())
print("dạng rộng sẽ có:", weather["month"].nunique(), "hàng x", weather["city"].nunique(), "cột")

dạng dài : (10, 3) (10 dòng, 3 cột)
số thành phố: 2 | số tháng: 5
dạng rộng sẽ có: 5 hàng x 2 cột


### 3.2. `nunique()`, `unique()` và `sorted`

`s.unique()` trả các **giá trị khác nhau** (theo thứ tự xuất hiện); `s.nunique()` đếm chúng. `sorted(...)` sắp xếp theo **thứ tự chữ cái**. Ta dùng chúng để biết bảng rộng có những hàng/cột nào.

In [3]:
print(weather["city"].unique().tolist())                 # theo thứ tự xuất hiện
print(sorted(weather["city"].unique()))                  # ['ElPaso', 'Jacksonville']
print(sorted(weather["month"].unique()))                 # ['April', 'February', 'January', 'March', 'May']

['Jacksonville', 'ElPaso']
['ElPaso', 'Jacksonville']
['April', 'February', 'January', 'March', 'May']


### 3.3. Từ điển lồng nhau `dict[hàng][cột]`

Ta có thể tự dựng bảng rộng bằng **từ điển lồng nhau**: `table[month][city] = temperature`. `dict.setdefault(key, {})` trả `table[key]` nếu đã có, nếu chưa thì tạo `{}` rồi trả về; nhờ vậy ta thêm giá trị không lo "chưa có khóa".

In [4]:
table = {}
table.setdefault("January", {})["ElPaso"] = 20
table.setdefault("January", {})["Jacksonville"] = 13
table.setdefault("May", {})["ElPaso"] = 43
print(table)                                      # {'January': {...}, 'May': {...}}
print(table["January"]["Jacksonville"])           # 13

{'January': {'ElPaso': 20, 'Jacksonville': 13}, 'May': {'ElPaso': 43}}
13


### 3.4. `pd.DataFrame.from_dict(..., orient="index")`

`from_dict(từ_điển_lồng, orient="index")` dựng bảng mà **khóa ngoài** (tháng) làm **nhãn hàng** và **khóa trong** (thành phố) làm **tên cột**. Ô nào không có dữ liệu thành `NaN`.

In [5]:
demo = pd.DataFrame.from_dict(table, orient="index")
print(demo)           # May không có Jacksonville -> NaN

         ElPaso  Jacksonville
January      20          13.0
May          43           NaN


### 3.5. `DataFrame.pivot(index=..., columns=..., values=...)`

- `index="month"`: giá trị cột `month` trở thành **nhãn hàng**;
- `columns="city"`: giá trị cột `city` trở thành **tên cột**;
- `values="temperature"`: giá trị đặt vào **ô**.

Kết quả được **sắp xếp** theo nhãn hàng và tên cột. Mỗi cặp (`month`, `city`) phải **duy nhất**, nếu không `pivot` báo lỗi. Từ pandas 2, các tham số phải truyền **theo tên** (keyword).

In [6]:
wide = weather.pivot(index="month", columns="city", values="temperature")
print(wide)
print("index.name  :", wide.index.name)           # month
print("columns.name:", wide.columns.name)         # city

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34
index.name  : month
columns.name: city


### 3.6. `reset_index()` — đưa nhãn hàng thành cột thường

Kết quả `pivot` có `month` làm **index** (không phải cột). `reset_index()` đưa nó về **cột thường** và đánh nhãn hàng `0..n-1`. Tên `city` còn lại làm "tên của trục cột" (không phải cột); đặt `rename_axis(columns=None)` để xóa tên đó nếu muốn bảng trông "sạch".

In [7]:
flat = wide.reset_index()
print(flat)
print(flat.columns.tolist())               # ['month', 'ElPaso', 'Jacksonville']
print(flat.rename_axis(columns=None).columns.name)   # None

city     month  ElPaso  Jacksonville
0        April       2             5
1     February       6            23
2      January      20            13
3        March      26            38
4          May      43            34
['month', 'ElPaso', 'Jacksonville']
None


### 3.7. `unstack()` — "mở" một cấp index thành cột

Nếu một Series có **index hai cấp** (`month`, `city`), `unstack()` đưa **cấp trong cùng** thành **cột**. Đây là cách làm khác dẫn đến cùng kết quả `pivot`. `set_index(["month", "city"])` tạo index hai cấp từ hai cột.

In [8]:
s = weather.set_index(["month", "city"])["temperature"]
print(s.head(4))
print(s.unstack("city").equals(wide))      # True: unstack cấp city -> cột

month     city        
January   Jacksonville    13
February  Jacksonville    23
March     Jacksonville    38
April     Jacksonville     5
Name: temperature, dtype: int64
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một **bảng đếm**: hàng là tháng, cột là thành phố. Với mỗi dòng của bảng dài, ta **tìm ô giao nhau** giữa hàng `month` và cột `city` rồi **ghi `temperature` vào đó**.

| dòng dài | city | month | temperature | ghi vào ô (hàng, cột) |
|---|---|---|---|---|
| 1 | Jacksonville | January | 13 | (January, Jacksonville) |
| 2 | Jacksonville | February | 23 | (February, Jacksonville) |
| 6 | ElPaso | January | 20 | (January, ElPaso) |
| 7 | ElPaso | February | 6 | (February, ElPaso) |
| ... | ... | ... | ... | ... |

Làm hết 10 dòng thì bảng `5 x 2` được lấp đầy. Sau đó xếp hàng và cột **theo thứ tự chữ cái** như pandas làm. Mọi cách ở dưới chỉ là cách khác nhau để thực hiện "tìm ô giao nhau rồi ghi giá trị".

## 5. Cách 1 — Brute Force / baseline: vòng lặp xây từ điển lồng nhau

**Ý tưởng.** Duyệt từng dòng, ghi `table[month][city] = temperature`. Dựng bảng bằng `from_dict(orient="index")`, rồi sắp xếp hàng/cột theo chữ cái và đặt tên cho trục như `pivot` làm.

In [9]:
def pivot_loop(weather: pd.DataFrame) -> pd.DataFrame:
    table = {}                                                    # table[month][city] = temperature
    for city, month, temp in zip(weather["city"], weather["month"], weather["temperature"]):
        table.setdefault(month, {})[city] = temp                  # tìm ô giao nhau rồi ghi
    out = pd.DataFrame.from_dict(table, orient="index")           # khóa ngoài = nhãn hàng
    out = out.sort_index().sort_index(axis=1)                     # hàng, cột theo chữ cái
    out.index.name = "month"
    out.columns.name = "city"
    return out


print(pivot_loop(weather))

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


In [10]:
res = pivot_loop(weather)
print("hình dạng:", res.shape)                         # (5, 2)
print("ElPaso   :", res["ElPaso"].tolist())            # [2, 6, 20, 26, 43]
print("Jacksonville:", res["Jacksonville"].tolist())   # [5, 23, 13, 38, 34]
print("bằng pivot chuẩn?", res.equals(weather.pivot(index="month", columns="city", values="temperature")))

hình dạng: (5, 2)
ElPaso   : [2, 6, 20, 26, 43]
Jacksonville: [5, 23, 13, 38, 34]
bằng pivot chuẩn? True


**Độ phức tạp (Cách 1).** Gọi `m` là số dòng dài, `r` số tháng, `k` số thành phố. Duyệt `m` dòng, mỗi dòng `O(1)` trung bình; dựng và sắp xếp bảng `r x k`: **Time `O(m + r log r + k log k)`** (cộng chi phí dựng `r·k` ô); từ điển lồng và bảng kết quả: **Space `O(m + r·k)`**.

**Nút thắt.** Phải tự lo sắp xếp, đặt tên trục, xử lý thiếu ô. pandas có `pivot` làm đúng việc này.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`weather.pivot(index="month", columns="city", values="temperature")`

- `index`: cột nào làm **nhãn hàng**;
- `columns`: cột nào làm **tên cột**;
- `values`: cột nào cho **giá trị ô**;
- kết quả tự **sắp xếp**; ô không có dữ liệu là `NaN`;
- mỗi cặp (`index`, `columns`) phải duy nhất.

Các cách khác: **`pivot_table`** (chịu được cặp trùng nhờ `aggfunc`, Cách 3), **`set_index(...).unstack()`** (Cách 4), **`groupby(...).sum().unstack()`** (Cách 5). Mọi cách đều là "cặp (hàng, cột) → một giá trị".

## 7. Các cách tiếp theo

### Cách 2 — `pivot(index, columns, values)` (cách chuẩn)

**Giải thích.** Một lời gọi, không sửa bảng đầu vào, trả bảng rộng có `month` là index và `city` là tên trục cột.

In [11]:
def pivot_pivot(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot(index="month", columns="city", values="temperature")


print(pivot_pivot(weather))

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def pivotTable(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot(index="month", columns="city", values="temperature")

**Độ phức tạp Cách 2:** **Time `O(m + r·k)`** (xếp `m` giá trị vào bảng `r x k`; cộng chi phí sắp xếp nhãn), **Space `O(r·k)`** (bảng kết quả).

### Cách 3 — `pivot_table(..., aggfunc="first")`

**Giải thích.** `pivot_table` giống `pivot` nhưng **cho phép cặp trùng** vì nó **gộp (aggregate)** các giá trị bằng `aggfunc` (mặc định `"mean"`, tức trung bình, biến kết quả thành `float`). Với dữ liệu mỗi cặp duy nhất của bài, `aggfunc="first"` (lấy giá trị đầu) giữ nguyên số nguyên.

In [13]:
def pivot_table_first(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot_table(index="month", columns="city", values="temperature", aggfunc="first")


print(pivot_table_first(weather))
print(weather.pivot_table(index="month", columns="city", values="temperature").dtypes.to_dict())   # mặc định mean -> float64

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34
{'ElPaso': dtype('float64'), 'Jacksonville': dtype('float64')}


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def pivotTable(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot_table(index="month", columns="city", values="temperature", aggfunc="first")

**Độ phức tạp Cách 3:** **Time `O(m)`** (gom nhóm) cộng sắp xếp nhãn, **Space `O(r·k)`**; hằng số lớn hơn `pivot`.

### Cách 4 — `set_index([...])` rồi `unstack()`

**Giải thích.** Đặt hai cột làm **index hai cấp**, lấy cột giá trị, rồi `unstack("city")` đưa cấp `city` lên làm cột. Đây là cơ chế bên trong của `pivot`; hữu ích khi dữ liệu đã nằm sẵn trong index hai cấp.

In [15]:
def pivot_unstack(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.set_index(["month", "city"])["temperature"].unstack("city")


print(pivot_unstack(weather))

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


**Độ phức tạp Cách 4:** **Time `O(m log m)`** (sắp xếp index hai cấp) hoặc gần `O(m)`, **Space `O(m + r·k)`**.

### Cách 5 — `groupby([...]).sum().unstack()`

**Giải thích.** Gom nhóm theo (`month`, `city`), lấy tổng (mỗi nhóm một dòng nên tổng = chính giá trị), rồi `unstack()`. Khác `pivot` ở chỗ **chịu cặp trùng** (cộng dồn). Cũng có `pd.crosstab(month, city, values=..., aggfunc="sum")` tương tự.

In [16]:
def pivot_groupby(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.groupby(["month", "city"])["temperature"].sum().unstack()


print(pivot_groupby(weather))

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


**Độ phức tạp Cách 5:** **Time `O(m)`** (băm nhóm) cộng sắp xếp, **Space `O(m + r·k)`**.

> Cách 2 là cách nên nộp. Cách 3 (`pivot_table`) hữu ích khi có thể có cặp trùng, nhớ chọn `aggfunc` phù hợp.

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết: từ điển lồng nhau lớn dần; (b) pipeline `pivot`: lấy các nhãn rồi lấp ô.

In [17]:
def trace_loop(weather):
    table = {}
    for step, (city, month, temp) in enumerate(zip(weather["city"], weather["month"], weather["temperature"]), start=1):
        table.setdefault(month, {})[city] = temp
        print(f"bước {step:2}: city={city:<12} month={month:<8} temp={temp:>2} -> table[{month!r}][{city!r}] = {temp}")
    print("\ntừ điển lồng cuối cùng:")
    for month in sorted(table):
        print("  ", month, table[month])
    out = pd.DataFrame.from_dict(table, orient="index").sort_index().sort_index(axis=1)
    out.index.name, out.columns.name = "month", "city"
    print("\nsau from_dict + sắp xếp:")
    print(out)
    return out


_ = trace_loop(weather)

bước  1: city=Jacksonville month=January  temp=13 -> table['January']['Jacksonville'] = 13
bước  2: city=Jacksonville month=February temp=23 -> table['February']['Jacksonville'] = 23
bước  3: city=Jacksonville month=March    temp=38 -> table['March']['Jacksonville'] = 38
bước  4: city=Jacksonville month=April    temp= 5 -> table['April']['Jacksonville'] = 5
bước  5: city=Jacksonville month=May      temp=34 -> table['May']['Jacksonville'] = 34
bước  6: city=ElPaso       month=January  temp=20 -> table['January']['ElPaso'] = 20
bước  7: city=ElPaso       month=February temp= 6 -> table['February']['ElPaso'] = 6
bước  8: city=ElPaso       month=March    temp=26 -> table['March']['ElPaso'] = 26
bước  9: city=ElPaso       month=April    temp= 2 -> table['April']['ElPaso'] = 2
bước 10: city=ElPaso       month=May      temp=43 -> table['May']['ElPaso'] = 43

từ điển lồng cuối cùng:
   April {'Jacksonville': 5, 'ElPaso': 2}
   February {'Jacksonville': 23, 'ElPaso': 6}
   January {'Jacksonvill

In [18]:
def trace_pivot(weather):
    print("BƯỚC 0 - bảng dài", weather.shape)
    print(weather.head(3))

    months = sorted(weather["month"].unique())
    cities = sorted(weather["city"].unique())
    print("\nBƯỚC 1 - nhãn hàng (month, theo chữ cái):", months)
    print("         tên cột (city, theo chữ cái):", cities)

    print("\nBƯỚC 2 - tra từng ô (hàng, cột):")
    for m in months:
        row = []
        for c in cities:
            hit = weather[(weather["month"] == m) & (weather["city"] == c)]["temperature"]
            row.append(int(hit.iloc[0]) if len(hit) == 1 else None)
        print(f"   {m:<9}", row)

    out = weather.pivot(index="month", columns="city", values="temperature")
    print("\nBƯỚC 3 - pivot trả về:", out.shape)
    print(out)
    return out


_ = trace_pivot(weather)

BƯỚC 0 - bảng dài (10, 3)
           city     month  temperature
0  Jacksonville   January           13
1  Jacksonville  February           23
2  Jacksonville     March           38

BƯỚC 1 - nhãn hàng (month, theo chữ cái): ['April', 'February', 'January', 'March', 'May']
         tên cột (city, theo chữ cái): ['ElPaso', 'Jacksonville']

BƯỚC 2 - tra từng ô (hàng, cột):
   April     [2, 5]
   February  [6, 23]
   January   [20, 13]
   March     [26, 38]
   May       [43, 34]

BƯỚC 3 - pivot trả về: (5, 2)
city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | bảng dài | `(10, 3)` |
| 1 | nhãn hàng / cột | 5 tháng và 2 thành phố, theo chữ cái |
| 2 | mỗi ô (tháng, thành phố) | đúng 1 dòng dài ứng với nó |
| 3 | `pivot` | bảng `(5, 2)`, `month` là index, `city` là tên cột |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def pivotTable(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot(index="month", columns="city", values="temperature")


print(pivotTable(weather))

city      ElPaso  Jacksonville
month                         
April          2             5
February       6            23
January       20            13
March         26            38
May           43            34


## 10. Vì sao thuật toán đúng

- **Mỗi dòng dài vào đúng một ô:** `pivot` đặt `temperature` của dòng (`city = c`, `month = m`) vào ô nằm ở hàng `m`, cột `c`.
- **Không đè, không mất:** vì mỗi cặp (`month`, `city`) **duy nhất** (đề cam kết), không có hai dòng cùng tranh một ô; do đó 10 dòng cho đúng 10 ô, và mọi giá trị của bảng gốc có mặt trong bảng rộng.
- **Đủ hàng, đủ cột:** số hàng = số tháng khác nhau, số cột = số thành phố khác nhau; ô không có dòng ứng với nó là `NaN` (với dữ liệu đủ như ví dụ thì không có).
- **Thứ tự:** pandas sắp nhãn hàng/cột theo thứ tự tăng dần (chữ cái với chuỗi), nên kết quả **xác định** và không phụ thuộc thứ tự dòng đầu vào.
- **Tính chất kiểm được:** với mỗi dòng `(c, m, t)` của bảng gốc, `result.loc[m, c] == t`.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số dòng của bảng dài, `r` số tháng khác nhau, `k` số thành phố khác nhau (`r·k` là số ô của bảng rộng).

- **Time:** `O(m)` để đặt giá trị vào ô cộng chi phí xác định và sắp xếp nhãn: khoảng `O(m + r log r + k log k)`. Cách 4 sắp xếp index hai cấp: `O(m log m)`.
- **Space:** `O(r·k)` cho bảng kết quả, cộng bộ nhớ phụ `O(m)` tuỳ cách.
- Nếu dữ liệu **thưa** (nhiều cặp không có), `r·k` có thể lớn hơn `m` nhiều và bảng rộng có nhiều `NaN`.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`. Các hàm không sửa bảng đầu vào. `pivot_table` mặc định (không `aggfunc`) cho `float` nên ta chỉ đưa bản `first` vào đối chiếu.

In [20]:
impls = {
    "loop": pivot_loop,
    "pivot": pivot_pivot,
    "pivot_table": pivot_table_first,
    "unstack": pivot_unstack,
    "groupby": pivot_groupby,
    "submit": pivotTable,
}

expected_example = pd.DataFrame(
    {"ElPaso": [2, 6, 20, 26, 43], "Jacksonville": [5, 23, 13, 38, 34]},
    index=pd.Index(["April", "February", "January", "March", "May"], name="month"),
)
expected_example.columns.name = "city"

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(weather), expected_example, check_column_type=False, check_index_type=False)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def make(rows):
    """rows: danh sách (city, month, temperature)."""
    return pd.DataFrame(rows, columns=["city", "month", "temperature"])


def check_all(df, expected_dict, label):
    """expected_dict[(month, city)] = temperature"""
    for key, fn in impls.items():
        out = fn(df)
        for (m, c), t in expected_dict.items():
            assert out.loc[m, c] == t, (label, key, m, c, out.loc[m, c], t)
        assert out.notna().sum().sum() == len(expected_dict), (label, key)
    print("PASS", label, "->", len(expected_dict), "ô có dữ liệu")


check_all(make([("A", "Jan", 5)]), {("Jan", "A"): 5}, "một dòng -> bảng 1x1")
check_all(make([("A", "Jan", 5), ("A", "Feb", 7)]), {("Jan", "A"): 5, ("Feb", "A"): 7}, "một thành phố, hai tháng")
check_all(make([("A", "Jan", 5), ("B", "Jan", 9)]), {("Jan", "A"): 5, ("Jan", "B"): 9}, "hai thành phố, một tháng")
check_all(make([("B", "Feb", 1), ("A", "Jan", 2), ("B", "Jan", 3)]),
          {("Feb", "B"): 1, ("Jan", "A"): 2, ("Jan", "B"): 3}, "thiếu ô (Feb, A) -> NaN")
check_all(make([("A", "Jan", -3), ("A", "Feb", 0)]), {("Jan", "A"): -3, ("Feb", "A"): 0}, "nhiệt độ âm và 0")

PASS một dòng -> bảng 1x1 -> 1 ô có dữ liệu
PASS một thành phố, hai tháng -> 2 ô có dữ liệu
PASS hai thành phố, một tháng -> 2 ô có dữ liệu
PASS thiếu ô (Feb, A) -> NaN -> 3 ô có dữ liệu
PASS nhiệt độ âm và 0 -> 2 ô có dữ liệu


### 12.2. Thứ tự dòng, ô thiếu, tên trục, đầu vào, và phản ví dụ

In [22]:
# Thứ tự dòng đầu vào không ảnh hưởng kết quả
shuffled = weather.sample(frac=1, random_state=0)
for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(shuffled), expected_example, check_column_type=False, check_index_type=False)
print("PASS đảo thứ tự dòng đầu vào -> kết quả y hệt")

# Ô thiếu thành NaN
gap = make([("A", "Jan", 5), ("B", "Feb", 9)])
for key, fn in impls.items():
    out = fn(gap)
    assert pd.isna(out.loc["Jan", "B"]) and pd.isna(out.loc["Feb", "A"]), key
print("PASS ô không có dữ liệu -> NaN")

# Tên trục của kết quả pivot, và dạng 'phẳng' sau reset_index
out = pivotTable(weather)
assert out.index.name == "month" and out.columns.name == "city"
flat = out.reset_index().rename_axis(columns=None)
assert flat.columns.tolist() == ["month", "ElPaso", "Jacksonville"]
print("PASS tên trục; reset_index cho cột month:", flat.columns.tolist())

# Đầu vào không bị sửa
before = weather.copy()
for key, fn in impls.items():
    fn(weather)
pd.testing.assert_frame_equal(weather, before)
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: cặp trùng làm pivot báo lỗi, pivot_table gộp được
dup = make([("A", "Jan", 5), ("A", "Jan", 7)])
try:
    pivotTable(dup)
    raise AssertionError("đáng lẽ phải lỗi")
except ValueError as e:
    print("PASS phản ví dụ: pivot báo ValueError ->", e)
assert dup.pivot_table(index="month", columns="city", values="temperature", aggfunc="sum").loc["Jan", "A"] == 12

# Phản ví dụ 2: đảo index và columns -> bảng chuyển vị
wrong = weather.pivot(index="city", columns="month", values="temperature")
assert wrong.shape == (2, 5) and wrong.T.shape == expected_example.shape
print("PASS phản ví dụ: đảo index/columns cho bảng", wrong.shape, "thay vì (5, 2)")

PASS đảo thứ tự dòng đầu vào -> kết quả y hệt
PASS ô không có dữ liệu -> NaN


PASS tên trục; reset_index cho cột month: ['month', 'ElPaso', 'Jacksonville']
PASS đầu vào không bị sửa
PASS phản ví dụ: pivot báo ValueError -> Index contains duplicate entries, cannot reshape


PASS phản ví dụ: đảo index/columns cho bảng (2, 5) thay vì (5, 2)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu ngẫu nhiên **mỗi cặp (tháng, thành phố) tối đa một dòng** và đủ ô, rồi so mọi cách với `pivot`. Sau đó thử dữ liệu thưa (có ô thiếu).

In [23]:
import random

random.seed(2889)
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
cities = ["A", "B", "C", "D"]

# Dữ liệu đủ ô
for trial in range(100):
    ms = random.sample(months, random.randint(1, 6))
    cs = random.sample(cities, random.randint(1, 4))
    rows = [(c, m, random.randint(-20, 45)) for c in cs for m in ms]
    random.shuffle(rows)
    df = make(rows)
    ref = df.pivot(index="month", columns="city", values="temperature")
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df), ref, check_column_type=False, check_index_type=False)
print("PASS 100 bảng ngẫu nhiên đủ ô - mọi cách khớp nhau")

# Dữ liệu thưa (bỏ ngẫu nhiên vài ô)
for trial in range(100):
    rows = [(c, m, random.randint(-20, 45)) for c in cities for m in months if random.random() < 0.6]
    if not rows:
        continue
    df = make(rows)
    ref = df.pivot(index="month", columns="city", values="temperature")
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df), ref, check_dtype=False, check_column_type=False, check_index_type=False)
print("PASS 100 bảng ngẫu nhiên thưa - mọi cách khớp nhau (kiểu có thể là float vì NaN)")

PASS 100 bảng ngẫu nhiên đủ ô - mọi cách khớp nhau


PASS 100 bảng ngẫu nhiên thưa - mọi cách khớp nhau (kiểu có thể là float vì NaN)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `pivot` khi có cặp trùng

In [24]:
dup = make([("A", "Jan", 5), ("A", "Jan", 7), ("A", "Feb", 1)])
try:
    dup.pivot(index="month", columns="city", values="temperature")
except ValueError as e:
    print("ValueError:", e)
print("pivot_table sum :", dup.pivot_table(index="month", columns="city", values="temperature", aggfunc="sum")["A"].tolist())
print("pivot_table mean:", dup.pivot_table(index="month", columns="city", values="temperature", aggfunc="mean")["A"].tolist())

ValueError: Index contains duplicate entries, cannot reshape
pivot_table sum : [1, 12]
pivot_table mean: [1.0, 6.0]


**Vì sao sai:** `pivot` không biết chọn giá trị nào khi hai dòng cùng (`month`, `city`). Đề bài này cam kết mỗi cặp duy nhất. Nếu dữ liệu có thể trùng, dùng `pivot_table` và **chọn `aggfunc` đúng ý** (tổng, trung bình, lớn nhất...).

### Lỗi 2: Truyền tham số theo vị trí

In [25]:
try:
    weather.pivot("month", "city", "temperature")
except TypeError as e:
    print("TypeError:", e)
print("đúng:", weather.pivot(index="month", columns="city", values="temperature").shape)

TypeError: DataFrame.pivot() takes 1 positional argument but 4 were given
đúng: (5, 2)


**Vì sao sai:** từ pandas 2, `pivot` chỉ nhận tham số **theo tên** (`index=`, `columns=`, `values=`). Hãy luôn viết đủ tên tham số cho rõ nghĩa.

### Lỗi 3: Nhầm `index` với `columns`

In [26]:
sai = weather.pivot(index="city", columns="month", values="temperature")
print("sai  :", sai.shape, "<- hàng là thành phố, cột là tháng (bảng chuyển vị)")
print(sai)
dung = weather.pivot(index="month", columns="city", values="temperature")
print("đúng :", dung.shape)

sai  : (2, 5) <- hàng là thành phố, cột là tháng (bảng chuyển vị)
month         April  February  January  March  May
city                                              
ElPaso            2         6       20     26   43
Jacksonville      5        23       13     38   34
đúng : (5, 2)


**Vì sao sai:** `index` quyết định cột nào thành **hàng**, `columns` quyết định cột nào thành **cột**. Đề muốn **mỗi hàng là một tháng** (`index="month"`) và **mỗi cột là một thành phố** (`columns="city"`). Đảo hai tham số cho ra bảng chuyển vị.

### Lỗi 4: Tưởng tháng được xếp theo thời gian

In [27]:
wide = pivotTable(weather)
print("thứ tự pandas:", wide.index.tolist())           # April, February, January, March, May
order = ["January", "February", "March", "April", "May"]
print("theo thời gian:", wide.reindex(order).index.tolist())

thứ tự pandas: ['April', 'February', 'January', 'March', 'May']
theo thời gian: ['January', 'February', 'March', 'April', 'May']


**Vì sao dễ nhầm:** `pivot` sắp nhãn chuỗi theo **bảng chữ cái** (`April` < `February` < `January`...), không phải theo lịch. Ví dụ của đề cũng xếp như vậy. Nếu cần thứ tự thời gian, `reindex(order)` hoặc dùng kiểu `Categorical` có thứ tự. Với đề này, giữ nguyên thứ tự pandas.

### Lỗi 5: Quên rằng `month` là index, không phải cột

In [28]:
wide = pivotTable(weather)
print("cột     :", wide.columns.tolist())              # chỉ ['ElPaso', 'Jacksonville']
try:
    wide["month"]
except KeyError as e:
    print("KeyError:", e)
print("sau reset_index:", wide.reset_index().columns.tolist())

cột     : ['ElPaso', 'Jacksonville']
KeyError: 'month'
sau reset_index: ['month', 'ElPaso', 'Jacksonville']


**Vì sao dễ nhầm:** sau `pivot`, `month` nằm ở **index**, không phải một cột. Muốn dùng như cột thì `reset_index()`. Đối chiếu đề gốc để biết dạng nào được yêu cầu: nếu đề hiển thị `month` như một cột, thêm `.reset_index()` (và có thể `.rename_axis(columns=None)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "xoay bảng / mỗi hàng là X, mỗi cột là Y / bảng chéo (cross-tab)" → nghĩ tới **`pivot`**.
- Công thức nhớ: **`df.pivot(index="hàng", columns="cột", values="giá trị")`**: "hàng nào, cột nào, ô là gì".
- Dữ liệu có thể **trùng cặp** → **`pivot_table(..., aggfunc=...)`**.
- Muốn đếm số lần xuất hiện: `pd.crosstab(df["a"], df["b"])`.
- Dài -> rộng: `pivot`; rộng -> dài: `melt` (bài sau).
- Muốn `month` thành cột thường: `.reset_index()`; muốn bỏ nhãn trục cột: `.rename_axis(columns=None)`.
- Luôn in `df.shape` trước/sau và đếm `NaN` để kiểm tra.

## 15. Biến thể của bài

1. Pivot nhiều giá trị: `df.pivot(index="month", columns="city", values=["temperature", "humidity"])`.
2. **Tổng hợp** khi có cặp trùng: `pivot_table(..., aggfunc="mean")`, `"max"`, `"sum"`.
3. Nhiều hàm cùng lúc: `pivot_table(..., aggfunc=["min", "max"])`.
4. **Điền ô thiếu**: `pivot_table(..., fill_value=0)` hoặc `pivot(...).fillna(0)`.
5. Thêm **tổng hàng/cột**: `pivot_table(..., margins=True)`.
6. Chuyển ngược lại (**melt**): `wide.reset_index().melt(id_vars="month", var_name="city", value_name="temperature")`.

Ta thử vài biến thể:

In [29]:
extra = weather.assign(humidity=weather["temperature"] * 2)
print("1.", extra.pivot(index="month", columns="city", values=["temperature", "humidity"]).shape)

dup = pd.concat([weather, weather.iloc[[0]].assign(temperature=99)], ignore_index=True)    # Jacksonville January lặp: 13 và 99
pt = dup.pivot_table(index="month", columns="city", values="temperature", aggfunc="max")
print("2. max khi trùng:", pt.loc["January", "Jacksonville"])                               # 99

sparse = weather.iloc[:7]
print("4. fill_value :", sparse.pivot_table(index="month", columns="city", values="temperature", fill_value=0, aggfunc="first").loc["April", "ElPaso"])
print("5. margins    :", weather.pivot_table(index="month", columns="city", values="temperature", aggfunc="sum", margins=True).shape)

back = pivotTable(weather).reset_index().melt(id_vars="month", var_name="city", value_name="temperature")
print("6. melt ngược :", back.shape)

1. (5, 4)
2. max khi trùng: 99
4. fill_value : 0
5. margins    : (6, 3)
6. melt ngược : (10, 3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + từ điển lồng | tự điền ô (hàng, cột) | `O(m + r log r + k log k)` (Python) | `O(m + r·k)` | hiểu bản chất |
| 2. `pivot(index, columns, values)` | xoay trực tiếp | `O(m + ...)` | `O(r·k)` | **cách nộp bài**; cặp duy nhất |
| 3. `pivot_table(..., aggfunc)` | xoay + gộp cặp trùng | `O(m)` + sắp xếp | `O(r·k)` | dữ liệu có thể trùng cặp; cần tổng hợp |
| 4. `set_index([...]).unstack()` | index hai cấp rồi mở cấp | `O(m log m)` | `O(m + r·k)` | dữ liệu đã ở dạng index hai cấp |
| 5. `groupby([...]).sum().unstack()` | gom nhóm rồi mở | `O(m)` + sắp xếp | `O(m + r·k)` | cần gộp trùng bằng tổng, nối với `groupby` khác |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `weather.pivot(index="month", columns="city", values="temperature").shape`, `weather["city"].nunique()`, `sorted(weather["month"].unique())[0]`, `weather.pivot(index="city", columns="month", values="temperature").shape`, `weather.pivot(index="month", columns="city", values="temperature").loc["May", "ElPaso"]`.

**Bài 2 (tính tay).** Bảng dài gồm các dòng `(city, month, temperature)`: `(A, x, 1)`, `(A, y, 2)`, `(B, x, 3)`. Viết bảng rộng của `pivot(index="city", columns="month", values="temperature")`; ô nào là `NaN`?

**Bài 3 (điền TODO).** Hoàn thành `pivot_todo(df)` bằng `pivot`.

In [30]:
def pivot_todo(df):
    # TODO: return df.pivot(index=..., columns=..., values=...)
    # gợi ý: hàng là month, cột là city, ô là temperature
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `pivot_loop2(weather)` bằng từ điển lồng nhau (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `pivot_flat(weather)` trả về bảng có `month` là **cột thường** (không phải index) và **không có tên trục cột**. (b) Viết `pivot_safe(df)` chịu được cặp trùng bằng cách lấy **nhiệt độ lớn nhất** của mỗi cặp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(5, 2)`, `2`, `'April'`, `(2, 5)`, `43`. Bài 2: hàng là `city`, cột là `month`.
- **Mức 2 (vừa):** Bài 2: hàng A: `x=1`, `y=2`; hàng B: `x=3`, `y=NaN` (không có dòng (B, y)). Bài 3: `index="month"`, `columns="city"`, `values="temperature"`. Bài 4: `table.setdefault(month, {})[city] = temp`, rồi `from_dict(orient="index")`, sắp xếp và đặt tên trục.
- **Mức 3 (sâu):** Bài 5(a): `weather.pivot(index="month", columns="city", values="temperature").reset_index().rename_axis(columns=None)`. Bài 5(b): `df.pivot_table(index="month", columns="city", values="temperature", aggfunc="max")`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert weather.pivot(index="month", columns="city", values="temperature").shape == (5, 2)
assert weather["city"].nunique() == 2
assert sorted(weather["month"].unique())[0] == "April"
assert weather.pivot(index="city", columns="month", values="temperature").shape == (2, 5)
assert weather.pivot(index="month", columns="city", values="temperature").loc["May", "ElPaso"] == 43

# Bài 2
t2 = pd.DataFrame({"city": ["A", "A", "B"], "month": ["x", "y", "x"], "temperature": [1, 2, 3]})
w2 = t2.pivot(index="city", columns="month", values="temperature")
assert w2.loc["A", "x"] == 1 and w2.loc["A", "y"] == 2 and w2.loc["B", "x"] == 3
assert pd.isna(w2.loc["B", "y"]) and w2.isna().sum().sum() == 1

# Bài 3
def pivot_todo_ans(df):
    return df.pivot(index="month", columns="city", values="temperature")

pd.testing.assert_frame_equal(pivot_todo_ans(weather), expected_example, check_column_type=False, check_index_type=False)

# Bài 4
def pivot_loop2(weather):
    table = {}
    for city, month, temp in zip(weather["city"], weather["month"], weather["temperature"]):
        table.setdefault(month, {})[city] = temp
    out = pd.DataFrame.from_dict(table, orient="index").sort_index().sort_index(axis=1)
    out.index.name, out.columns.name = "month", "city"
    return out

pd.testing.assert_frame_equal(pivot_loop2(weather), expected_example, check_column_type=False, check_index_type=False)

# Bài 5(a)
def pivot_flat(weather):
    return weather.pivot(index="month", columns="city", values="temperature").reset_index().rename_axis(columns=None)

f = pivot_flat(weather)
assert f.columns.tolist() == ["month", "ElPaso", "Jacksonville"] and f.columns.name is None
assert f["month"].tolist() == ["April", "February", "January", "March", "May"]

# Bài 5(b)
def pivot_safe(df):
    return df.pivot_table(index="month", columns="city", values="temperature", aggfunc="max")

d = make([("A", "Jan", 5), ("A", "Jan", 7), ("B", "Jan", 1)])
assert pivot_safe(d).loc["Jan", "A"] == 7 and pivot_safe(d).loc["Jan", "B"] == 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xoay bảng dài -> rộng)

```python
import pandas as pd

def pivotTable(weather: pd.DataFrame) -> pd.DataFrame:
    return weather.pivot(index="month", columns="city", values="temperature")
```

**Checklist**

- [ ] `index="month"` (hàng), `columns="city"` (cột), `values="temperature"` (ô): không đảo nhầm?
- [ ] Truyền tham số **theo tên** (pandas 2+ không nhận vị trí)?
- [ ] Mỗi cặp (`month`, `city`) duy nhất? Nếu có thể trùng, dùng `pivot_table` với `aggfunc` rõ ràng?
- [ ] Biết `month` là **index**; cần cột thường thì `reset_index()` (đối chiếu đề gốc)?
- [ ] Biết nhãn được **sắp theo chữ cái**, không theo thời gian?
- [ ] **Trả về** DataFrame; bảng đầu vào không bị sửa?
- [ ] Đã thử bảng một dòng và dữ liệu thiếu ô (kết quả có `NaN`)?
- [ ] Skeleton đúng `def pivotTable(weather: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?